# FitSense AI: análisis reproducible
Este notebook lee los CSV locales, integra por persona-fecha, resume el EDA, estandariza variables y ejecuta las cuatro alternativas. No depende de Google Colab ni genera observaciones ficticias.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.config import DATA_ORIGINAL, DEFAULT_FEATURES
from src.integration import run_integration
from src.eda import descriptive_summary, missing_summary, findings
from src.preprocessing import prepare_features
from src.kmeans_model import search_k, fit_kmeans
from src.dbscan_model import fit_dbscan, k_distances
from src.pca_model import fit_pca
from src.combined_models import fit_pca_kmeans, fit_pca_dbscan
from src.model_comparison import compare_models

In [ ]:
integrated, audit = run_integration(DATA_ORIGINAL / 'dailyActivity_merged.csv', DATA_ORIGINAL / 'sleepDay_merged.csv', ROOT)
display(audit)
display(integrated.head())

In [ ]:
display(descriptive_summary(integrated).round(2))
display(missing_summary(integrated))
findings(integrated)

In [ ]:
features = [c for c in DEFAULT_FEATURES if c in integrated.columns]
prepared = prepare_features(integrated, features, missing='median', outliers='keep')
X = prepared['scaled'].to_numpy()
print(features, X.shape)

In [ ]:
k_results = search_k(X, range(2, min(11, len(X))))
display(k_results)
kmeans, labels_km = fit_kmeans(X, k=3)
dbscan, labels_db, db_metrics, kinds = fit_dbscan(X, eps=0.8, min_samples=5)
pca, scores, pca_info = fit_pca(X, variance_threshold=0.90)
km_pca, labels_kmp = fit_pca_kmeans(scores, k=3)
db_pca, labels_dbp, db_pca_metrics, kinds_pca = fit_pca_dbscan(scores, eps=0.8, min_samples=5)
display(pca_info['loadings']); print('PCA', pca_info['variance_retained'], pca_info['reconstruction_rmse_scaled'])

In [ ]:
display(compare_models(X, {'K-Means': labels_km, 'DBSCAN': labels_db, 'PCA + K-Means': labels_kmp, 'PCA + DBSCAN': labels_dbp}))
print('DBSCAN ruido:', int((labels_db == -1).sum()), 'de', len(labels_db))
print('La interpretación final debe considerar cobertura, estabilidad y significado en las unidades originales; no solo Silhouette.')